In [9]:
# Load and clean dataset
df = preprocessing.load_data(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# Create engineered feature
df = preprocessing.create_features(df)

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

df.head()

Dataset shape: (7032, 21)
Missing values: 0


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,num_services
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0,1
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,0,3
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0,3
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,1


In [10]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5625, 20)
X_test shape: (1407, 20)
y_train shape: (5625,)
y_test shape: (1407,)


In [11]:
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Create 5 machine learning models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    ),

    "SVM": SVC(
        probability=True,
        class_weight="balanced",
        random_state=42
    )
}

# Create a full pipeline for each model
model_pipelines = {
    name: Pipeline([
        ("preprocessor", preprocessing.create_preprocessor(df)),
        ("model", model)
    ])
    for name, model in models.items()
}

print("Total models:", len(model_pipelines))
print("\nModels created successfully:")
for name in model_pipelines:
    print("-", name)

Total models: 5

Models created successfully:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting
- SVM


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Create 5-fold stratified cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_results = {}

for name, pipeline in model_pipelines.items():
    print(f"Evaluating: {name}")

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results[name] = scores

    print("AUC scores:", scores.round(4))
    print("Mean AUC:", round(scores.mean(), 4))
    print("-" * 40)

Evaluating: Logistic Regression
AUC scores: [0.8468 0.843  0.8374 0.8496 0.853 ]
Mean AUC: 0.8459
----------------------------------------
Evaluating: Decision Tree
AUC scores: [0.6328 0.6235 0.6765 0.6658 0.6507]
Mean AUC: 0.6499
----------------------------------------
Evaluating: Random Forest
AUC scores: [0.8281 0.8113 0.816  0.8274 0.8297]
Mean AUC: 0.8225
----------------------------------------
Evaluating: Gradient Boosting
AUC scores: [0.8428 0.8456 0.8499 0.8524 0.8511]
Mean AUC: 0.8484
----------------------------------------
Evaluating: SVM
AUC scores: [0.8237 0.8226 0.8196 0.8299 0.8381]
Mean AUC: 0.8268
----------------------------------------


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

results = []

for name, pipeline in model_pipelines.items():
    print(f"\nTraining: {name}")

    # Train model on training data
    pipeline.fit(X_train, y_train)

    # Predict on unseen test data
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Calculate metrics
    results.append({
        "Model": name,
        "CV AUC Mean": cv_results[name].mean(),
        "CV AUC Std": cv_results[name].std(),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Test Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "Test F1": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "Test ROC-AUC": roc_auc_score(y_test, y_prob)
    })

    print(classification_report(
        y_test,
        y_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    ))


Training: Logistic Regression
              precision    recall  f1-score   support

    No Churn       0.90      0.70      0.79      1033
       Churn       0.49      0.80      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.74      1407


Training: Decision Tree
              precision    recall  f1-score   support

    No Churn       0.82      0.81      0.81      1033
       Churn       0.48      0.50      0.49       374

    accuracy                           0.73      1407
   macro avg       0.65      0.65      0.65      1407
weighted avg       0.73      0.73      0.73      1407


Training: Random Forest
              precision    recall  f1-score   support

    No Churn       0.82      0.89      0.85      1033
       Churn       0.60      0.47      0.53       374

    accuracy                           0.78      1407
   macro avg       0.71      0.68      0.69    

In [14]:
# Create comparison DataFrame
results_df = pd.DataFrame(results)

# Sort models by mean cross-validation AUC
results_df = results_df.sort_values(
    by="CV AUC Mean",
    ascending=False
).reset_index(drop=True)

# Display rounded results
results_df = results_df.round(4)

results_df

,Model,CV AUC Mean,CV AUC Std,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC
0,Gradient Boosting,0.8484,0.0036,0.7960,0.6417,0.5267,0.5786,0.8384
1,Logistic Regression,0.8459,0.0054,0.7257,0.4901,0.7968,0.6069,0.8351
2,SVM,0.8268,0.0066,0.7271,0.4915,0.7727,0.6008,0.8093
3,Random Forest,0.8225,0.0074,0.7775,0.6041,0.4733,0.5307,0.8110
4,Decision Tree,0.6499,0.0197,0.7257,0.4844,0.4973,0.4908,0.6538


In [15]:
best_model_name = results_df.loc[0, "Model"]

print("Best Model:", best_model_name)
print("Mean CV AUC:", results_df.loc[0, "CV AUC Mean"])
print("Test ROC-AUC:", results_df.loc[0, "Test ROC-AUC"])

Best Model: Gradient Boosting
Mean CV AUC: 0.8484
Test ROC-AUC: 0.8384


## Model Evaluation and Conclusion

Five machine learning models were evaluated using
5-fold Stratified Cross-Validation and an unseen test dataset.

The models evaluated were:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. Gradient Boosting
5. Support Vector Machine (SVM)

Gradient Boosting achieved the highest mean cross-validation
ROC-AUC score of 0.8484 and a test ROC-AUC score of 0.8384.

Therefore, Gradient Boosting was selected as the best model
based on mean cross-validation ROC-AUC.

The model can help identify customers who may be at risk
of leaving the telecom service. Further evaluation of recall,
precision, and the business cost of missed churn cases is
recommended before real-world deployment.